# Custom RAG Evaluation with Integrated Ragas Metrics

This notebook evaluates all 4 embedding models with:
1. **Custom RAG Pipeline**: Retrieve → Rerank → Generate
2. **Integrated Evaluation**: Calculate Ragas metrics per-query
3. **Real-time Monitoring**: Track progress and metrics
4. **Multi-model Comparison**: Test all embedding models

## Pipeline Flow:
```
Question → Embed → Similarity Search (top-10) → Rerank (top-5) → Generate Answer → Ragas Metrics
```

In [ ]:
# Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

import warnings
warnings.filterwarnings('ignore')

print("✓ Setup complete")

In [ ]:
# Imports
import json
import time
from datetime import datetime
from typing import List, Dict, Any

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import plotly.graph_objects as go

# LangChain
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import PromptTemplate

# Ragas
from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy
)
from ragas.llms import LangchainLLMWrapper

# Reranking
from flashrank import Ranker, RerankRequest

# Config
from custom_rag_config import (
    EMBEDDING_MODELS,
    GENERATION_LLM,
    RETRIEVAL_CONFIG,
    VECTOR_STORE_CONFIG,
    RAG_PROMPT_TEMPLATE,
    RAGAS_EVALUATOR_MODEL,
    TEST_DATASET_PATH,
    RESULTS_DIR,
    get_reranker_config
)
from utils import load_dataset_from_jsonl

print("✓ All imports successful")

## Step 1: Load Test Dataset

In [ ]:
# Load test questions
test_dataset = load_dataset_from_jsonl(TEST_DATASET_PATH)

print(f"📊 Loaded {len(test_dataset)} test questions")
print(f"\nFirst 3 questions:")
for i, item in enumerate(test_dataset[:3]):
    print(f"{i+1}. {item['user_input']}")

print(f"\n✓ Test dataset ready")

## Step 2: Initialize Components

In [ ]:
# Initialize LLM for answer generation
llm = ChatOpenAI(
    model=GENERATION_LLM["model"],
    temperature=GENERATION_LLM["temperature"],
    max_tokens=GENERATION_LLM["max_tokens"]
)

print(f"✓ Generation LLM: {GENERATION_LLM['model']}")

# Initialize Ragas evaluator LLM
evaluator_llm = ChatOpenAI(model=RAGAS_EVALUATOR_MODEL, temperature=0)
evaluator_llm_wrapper = LangchainLLMWrapper(evaluator_llm)

print(f"✓ Ragas evaluator: {RAGAS_EVALUATOR_MODEL}")

# Initialize Ragas metrics
ragas_metrics = [
    ContextPrecision(llm=evaluator_llm_wrapper),
    ContextRecall(llm=evaluator_llm_wrapper),
    Faithfulness(llm=evaluator_llm_wrapper),
    AnswerRelevancy(llm=evaluator_llm_wrapper)
]

print(f"✓ Ragas metrics initialized")

# Initialize reranker
if RETRIEVAL_CONFIG["use_reranking"]:
    reranker_config = get_reranker_config()
    reranker = Ranker(model_name=reranker_config["model"])
    print(f"✓ Reranker: {reranker_config['display_name']}")
else:
    reranker = None
    print(f"⚠️  Reranking disabled")

# Initialize prompt template
prompt_template = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

print(f"\n✅ All components initialized")

## Step 3: Define RAG Pipeline Functions

In [ ]:
def retrieve_documents(vector_store, query: str, k: int = 10):
    """Retrieve top-k documents from vector store"""
    results = vector_store.similarity_search_with_score(query, k=k)
    documents = [doc for doc, score in results]
    scores = [float(score) for doc, score in results]
    return documents, scores


def rerank_documents(query: str, documents: List, reranker, top_n: int = 5):
    """Rerank documents using FlashRank"""
    if reranker is None:
        return documents[:top_n], [1.0] * min(top_n, len(documents))
    
    # Prepare passages
    passages = [
        {"id": i, "text": doc.page_content}
        for i, doc in enumerate(documents)
    ]
    
    # Rerank
    rerank_request = RerankRequest(query=query, passages=passages)
    reranked_results = reranker.rerank(rerank_request)
    
    # Get top-n after reranking
    top_results = reranked_results[:top_n]
    reranked_docs = [documents[result["id"]] for result in top_results]
    rerank_scores = [result["score"] for result in top_results]
    
    return reranked_docs, rerank_scores


def generate_answer(query: str, contexts: List[str], llm, prompt_template) -> str:
    """Generate answer using LLM"""
    # Format context
    context_str = "\n\n".join([
        f"[{i+1}] {ctx}" for i, ctx in enumerate(contexts)
    ])
    
    # Create prompt
    prompt = prompt_template.format(context=context_str, question=query)
    
    # Generate
    response = llm.invoke(prompt)
    return response.content


def rag_pipeline(
    query: str,
    vector_store,
    reranker,
    llm,
    prompt_template,
    top_k: int = 10,
    top_n_after_rerank: int = 5
) -> Dict[str, Any]:
    """Complete RAG pipeline: Retrieve → Rerank → Generate"""
    
    # Step 1: Retrieve
    documents, retrieval_scores = retrieve_documents(vector_store, query, k=top_k)
    
    # Step 2: Rerank
    reranked_docs, rerank_scores = rerank_documents(
        query, documents, reranker, top_n=top_n_after_rerank
    )
    
    # Step 3: Generate answer
    contexts = [doc.page_content for doc in reranked_docs]
    answer = generate_answer(query, contexts, llm, prompt_template)
    
    return {
        "retrieved_contexts": contexts,
        "retrieval_scores": retrieval_scores[:top_n_after_rerank],
        "rerank_scores": rerank_scores,
        "answer": answer,
        "num_retrieved": len(documents),
        "num_reranked": len(reranked_docs)
    }


print("✓ RAG pipeline functions defined")

## Step 4: Load Vector Stores

In [ ]:
def get_embedding_function(model_config):
    """Get embedding function for a model"""
    provider = model_config["provider"]
    model_name = model_config["model"]
    
    if provider == "openai":
        return OpenAIEmbeddings(model=model_name)
    elif provider == "huggingface":
        return HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
    else:
        raise ValueError(f"Unknown provider: {provider}")


# Load all vector stores
vector_stores = {}
persist_dir = VECTOR_STORE_CONFIG["persist_directory"]

print(f"📦 Loading vector stores from: {persist_dir}\n")

for model_config in EMBEDDING_MODELS:
    model_name = model_config["name"]
    collection_name = model_config["collection_name"]
    
    try:
        embedding_func = get_embedding_function(model_config)
        
        vector_store = Chroma(
            collection_name=collection_name,
            embedding_function=embedding_func,
            persist_directory=persist_dir
        )
        
        # Test vector store
        count = vector_store._collection.count()
        
        vector_stores[model_name] = vector_store
        print(f"✓ {model_config['display_name']:40s} ({count} chunks)")
        
    except Exception as e:
        print(f"✗ {model_config['display_name']:40s} FAILED: {e}")

print(f"\n✅ Loaded {len(vector_stores)} vector stores")

## Step 5: Run Evaluation for All Models

This will:
1. For each embedding model
2. For each test question
3. Run RAG pipeline
4. Calculate Ragas metrics immediately
5. Save results

In [ ]:
# Create results directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = Path(RESULTS_DIR) / f"evaluation_{timestamp}"
results_dir.mkdir(parents=True, exist_ok=True)

print(f"💾 Results will be saved to: {results_dir}")
print(f"\n{'='*80}")
print(f"STARTING MULTI-MODEL EVALUATION")
print(f"{'='*80}")
print(f"Models: {len(vector_stores)}")
print(f"Questions: {len(test_dataset)}")
print(f"Total queries: {len(vector_stores) * len(test_dataset)}")
print(f"{'='*80}\n")

In [ ]:
# Store all results
all_results = {}

for model_config in EMBEDDING_MODELS:
    model_name = model_config["name"]
    
    if model_name not in vector_stores:
        print(f"⚠️  Skipping {model_name} - vector store not loaded\n")
        continue
    
    print(f"\n{'='*80}")
    print(f"📊 EVALUATING: {model_config['display_name']}")
    print(f"{'='*80}\n")
    
    vector_store = vector_stores[model_name]
    eval_data = []
    
    # Progress bar
    pbar = tqdm(test_dataset, desc=f"{model_config['display_name']}", leave=True)
    
    for test_case in pbar:
        query = test_case['user_input']
        pbar.set_postfix_str(f"Query: {query[:40]}...")
        
        try:
            # Run RAG pipeline
            rag_result = rag_pipeline(
                query=query,
                vector_store=vector_store,
                reranker=reranker,
                llm=llm,
                prompt_template=prompt_template,
                top_k=RETRIEVAL_CONFIG["top_k"],
                top_n_after_rerank=RETRIEVAL_CONFIG["top_n_after_rerank"]
            )
            
            # Prepare data for Ragas
            eval_data.append({
                'user_input': query,
                'retrieved_contexts': rag_result['retrieved_contexts'],
                'response': rag_result['answer'],
                'reference': test_case['reference'],
                'reference_contexts': test_case['reference_contexts']
            })
            
        except Exception as e:
            print(f"\n❌ Error on query '{query[:50]}...': {e}")
            continue
        
        time.sleep(0.2)  # Small delay to avoid rate limits
    
    print(f"\n✓ Completed {len(eval_data)} queries")
    
    # Run Ragas evaluation
    if len(eval_data) > 0:
        print(f"\n⏳ Calculating Ragas metrics...")
        
        try:
            dataset = EvaluationDataset.from_list(eval_data)
            ragas_results = evaluate(dataset, metrics=ragas_metrics)
            
            # Convert to DataFrame
            results_df = ragas_results.to_pandas()
            
            # Calculate summary statistics
            summary = results_df.mean()
            
            print(f"\n📊 RESULTS for {model_config['display_name']}:")
            print(f"{'='*80}")
            for metric, value in summary.items():
                print(f"{metric:25s}: {value:.3f}")
            print(f"{'='*80}")
            
            # Save results
            model_results_dir = results_dir / model_name
            model_results_dir.mkdir(exist_ok=True)
            
            results_df.to_csv(model_results_dir / "results.csv", index=False)
            summary.to_frame(name='score').to_csv(model_results_dir / "summary.csv")
            
            # Store for comparison
            all_results[model_name] = {
                'config': model_config,
                'results_df': results_df,
                'summary': summary.to_dict(),
                'eval_data': eval_data
            }
            
            print(f"\n💾 Saved to: {model_results_dir}")
            
        except Exception as e:
            print(f"\n❌ Ragas evaluation failed: {e}")
    else:
        print(f"\n⚠️  No successful queries - skipping Ragas evaluation")

print(f"\n\n{'='*80}")
print(f"✅ EVALUATION COMPLETE")
print(f"{'='*80}")
print(f"Evaluated {len(all_results)} models")
print(f"Results saved to: {results_dir}")

## Step 6: Quick Comparison

In [ ]:
# Create comparison table
comparison_data = []

for model_name, result in all_results.items():
    row = {
        'Model': result['config']['display_name'],
        'Provider': result['config']['provider'],
        'Dimensions': result['config']['dimensions'],
    }
    row.update(result['summary'])
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

# Calculate overall average
metric_cols = [col for col in comparison_df.columns if col not in ['Model', 'Provider', 'Dimensions']]
comparison_df['Overall'] = comparison_df[metric_cols].mean(axis=1)

print("\n" + "="*100)
print("QUICK COMPARISON - All Embedding Models")
print("="*100)
print(comparison_df.round(3).to_string(index=False))
print("="*100)

# Save comparison
comparison_df.to_csv(results_dir / "comparison.csv", index=False)
print(f"\n💾 Saved comparison to: {results_dir / 'comparison.csv'}")

## Step 7: Quick Visualization

In [ ]:
# Bar chart comparison
fig = go.Figure()

for metric in metric_cols:
    fig.add_trace(go.Bar(
        name=metric,
        x=comparison_df['Model'],
        y=comparison_df[metric],
        text=comparison_df[metric].round(3),
        textposition='auto'
    ))

fig.update_layout(
    title='RAG Performance: Embedding Model Comparison',
    xaxis_title='Embedding Model',
    yaxis_title='Score',
    yaxis_range=[0, 1],
    barmode='group',
    height=600,
    showlegend=True
)

fig.show()

# Save
fig.write_html(str(results_dir / "comparison.html"))
print(f"\n💾 Saved chart to: {results_dir / 'comparison.html'}")

## Summary

✅ **Evaluation Complete!**

**Next Steps:**
1. Review the comparison table above
2. Open `custom_rag_03_comparison.ipynb` for detailed analysis
3. Check saved results in the results directory